In [18]:
import pandas as pd
import numpy as np

In [19]:
from sklearn.datasets import load_breast_cancer
breast_cancer = load_breast_cancer()
X, y = breast_cancer.data, breast_cancer.target

### Собственная реализация случайного леса

#### Будем использовать собственную реализацию решающего дерева в качестве базовой модели

In [32]:
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin

# ---- Вспомогательные функции (теперь вне класса) ----
def gini_criteria(total_by_class, total):
    if total == 0:
        return 0.0
    parts = total_by_class / total
    return 1 - np.sum(parts ** 2)

def best_split_for_feature(sorted_features, sorted_labels, parent_score, n_classes, min_samples_leaf, criteria):
    n_samples = len(sorted_features)
    best_threshold = None
    best_split = parent_score
    left_border = 0

    right_counts = np.bincount(sorted_labels, minlength=n_classes)
    right_volume = n_samples
    left_counts = np.zeros_like(right_counts)
    left_volume = 0

    while left_border < n_samples - 1:
        start_value = sorted_features[left_border]
        j = left_border
        while j < n_samples and sorted_features[j] == start_value:
            cls = sorted_labels[j]
            right_counts[cls] -= 1
            left_counts[cls] += 1
            right_volume -= 1
            left_volume += 1
            j += 1

        if left_volume >= min_samples_leaf and right_volume >= min_samples_leaf:
            criteria_left = criteria(left_counts, left_volume)
            criteria_right = criteria(right_counts, right_volume)
            current_score = left_volume / n_samples * criteria_left + right_volume / n_samples * criteria_right

            if current_score < best_split:
                best_split = current_score
                if j < n_samples:
                    best_threshold = (sorted_features[j] + start_value) / 2.0
                else:
                    best_threshold = start_value

        left_border = j
    return best_threshold, best_split

# ---- Класс узла с поддержкой pickle ----
class Node:
    __slots__ = ('feature', 'threshold', 'left', 'right', 'value')
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

    def __getstate__(self):
        return {slot: getattr(self, slot) for slot in self.__slots__}

    def __setstate__(self, state):
        for slot, value in state.items():
            setattr(self, slot, value)

# ---- Класс дерева ----
class MyDecisionTreeClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, max_depth=5, fit_intercept = True, min_samples_split=2, min_samples_leaf=1):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.root = None
        self.n_classes = None
        self.fit_intercept = fit_intercept

    def find_best_split(self, X, y, indices, parent_score):
        best_feature, best_threshold = None, None
        best_split = parent_score
        n_samples = len(indices)
        labels = y[indices]

        if n_samples < self.min_samples_leaf:
            return best_feature, best_threshold

        for feature in range(X.shape[1]):
            feature_values = X[indices, feature]
            sorted_idx = np.argsort(feature_values)
            sorted_features = feature_values[sorted_idx]
            sorted_labels = labels[sorted_idx]

            current_threshold, current_split = best_split_for_feature(
                sorted_features, sorted_labels, parent_score,
                self.n_classes, self.min_samples_leaf, gini_criteria
            )
            if current_split < best_split:
                best_feature = feature
                best_threshold = current_threshold
                best_split = current_split
        return best_feature, best_threshold

    def build_tree(self, X, y, indices, depth):
        n_samples = len(indices)
        # Определяем классы в текущем подмножестве
        unique_classes = np.unique(y[indices])
        n_classes_here = len(unique_classes)

        if (n_classes_here == 1) or (depth >= self.max_depth) or (n_samples < self.min_samples_split):
            most_common = np.bincount(y[indices]).argmax()
            return Node(value=most_common)

        parent_counts = np.bincount(y[indices], minlength=self.n_classes)
        parent_score = gini_criteria(parent_counts, n_samples)

        best_feature, best_threshold = self.find_best_split(X, y, indices, parent_score)

        if best_feature is None:
            most_common = np.bincount(y[indices]).argmax()
            return Node(value=most_common)

        feature_values = X[indices, best_feature]
        left_indices = indices[feature_values <= best_threshold]
        right_indices = indices[feature_values > best_threshold]

        left_subtree = self.build_tree(X, y, left_indices, depth + 1)
        right_subtree = self.build_tree(X, y, right_indices, depth + 1)

        return Node(feature=best_feature, threshold=best_threshold,
                    left=left_subtree, right=right_subtree)

    def fit(self, X, y):
        X, y = np.array(X), np.array(y)
        self.n_classes = len(np.unique(y))
        self.root = self.build_tree(X, y, indices=np.arange(X.shape[0]), depth=0)
        return self

    def predict_one(self, obj, node):
        if node.value is not None:
            return node.value
        if obj[node.feature] <= node.threshold:
            return self.predict_one(obj, node.left)
        return self.predict_one(obj, node.right)

    def predict(self, X):
        X = np.array(X)
        return np.array([self.predict_one(x, self.root) for x in X])

#### Построение ансамбля

In [33]:
from sklearn.base import BaseEstimator, RegressorMixin
from joblib import Parallel, delayed

class My_RandomForestClassifier(BaseEstimator, RegressorMixin):
    def __init__(self, n_estimators = 5, bootstrap = True, max_samples = 0.8, max_features = 0.8, fit_intercept = True, random_state = 76, n_jobs = -1):
        self.n_estimators = n_estimators 
        self.bootstrap = bootstrap
        self.max_samples = max_samples 
        self.max_features = max_features 
        self.fit_intercept = fit_intercept
        self.random_state = random_state
        self.rng = np.random.RandomState(random_state)
        self.estimators_ = []
        self.feature_indices_ = []
        self.n_jobs = n_jobs
    
        
    def bootstrap_samples(self, X, y):
        n_rows, n_cols = X.shape
        
        bootstrap_size = int(n_rows * self.max_samples)
        bootstrap_cols = int(n_cols * self.max_features)
        
        samples = []
        
        for _ in range(self.n_estimators):
            row_indices = self.rng.choice(
                n_rows, 
                size = bootstrap_size, 
                replace = True
            )

            cols_indices = self.rng.choice(
                n_cols,
                size = bootstrap_cols,
                replace = False
            )

            self.feature_indices_.append(cols_indices)
            
            
            X_sample = X[row_indices][:, cols_indices]
            y_sample = y[row_indices]
            
            samples.append((X_sample, y_sample, cols_indices))
            
        return samples

    def _fit_estimator(self, X_sample, y_sample, col_indices):
        model = MyDecisionTreeClassifier(fit_intercept = self.fit_intercept)
        model.fit(X_sample, y_sample)
        return model, col_indices
    
    def fit(self, X, y):
        self.n_samples = X.shape[0]
        self.classes_ = np.unique(y)
        cols = np.arange(X.shape[1])
        base_models_data = [(X, y, cols) for _ in range(self.n_estimators)]
        if self.bootstrap:
            base_models_data = self.bootstrap_samples(X, y)
        else:
            self.feature_indices_ = [np.arange(X.shape[1]) for _ in range(self.n_estimators)]

        results = Parallel(n_jobs=self.n_jobs)(
            delayed(self._fit_estimator)(X_s, y_s, cols)
            for (X_s, y_s, cols) in base_models_data
        )

        self.estimators_ = [res[0] for res in results]
        self.feature_indices_ = [res[1] for res in results]

        return self
        
    def predict(self, X):
        n_samples = X.shape[0]
        n_classes = len(self.classes_)
        predictions = np.zeros((len(self.estimators_), n_samples), dtype = int)
        
        for i, (model, feature_indices) in enumerate(zip(self.estimators_, self.feature_indices_)):
            X_sub = X[:, feature_indices]
            predictions[i, :] = model.predict(X_sub)

        one_hot = np.eye(n_classes)[predictions]
        vote_sum = one_hot.sum(axis = 0)
        return self.classes_[np.argmax(vote_sum, axis = 1)]

## Сравнение по качеству

In [34]:
from sklearn.model_selection import cross_val_score

### Моя реализация

In [35]:
%%time
my_random_forest_classifier = My_RandomForestClassifier(
    n_estimators = 5,
    bootstrap=True,        
    random_state=42,
    n_jobs=-1
)
print(f'Accuracy: {np.mean(cross_val_score(my_random_forest_classifier, X, y, cv = 7, scoring = 'accuracy'))}')

Accuracy: 0.9455843764786852
CPU times: total: 609 ms
Wall time: 15.7 s


### Готовая реализация sklearn

In [36]:
%%time
from sklearn.ensemble import RandomForestClassifier

random_forest_classifier = RandomForestClassifier(
    n_estimators = 50, 
    max_depth = 5,
    bootstrap=True,        
    random_state=42,
    n_jobs=-1    
)

print(f'Accuracy: {np.mean(cross_val_score(random_forest_classifier, X, y, cv = 7, scoring = 'accuracy'))}')

Accuracy: 0.9649202047576031
CPU times: total: 2.16 s
Wall time: 1.55 s
